In [1]:
import pandas as pd
import sys
sys.path.append('..')

from data.emission_factors import SCOPE1, SCOPE2_ELECTRICITY, BIOMETHANE
from data.sites import LACQ, FRENCH_GAS_SECTOR_AVG

print("Lacq Industrial Site — Full Decarbonization Case Study")
print("=" * 55)
print(f"Site:     {LACQ['name']}")
print(f"Sector:   {LACQ['sector']}")
print(f"Location: {LACQ['location']}")
print(f"Source:   {LACQ['note']}")

Lacq Industrial Site — Full Decarbonization Case Study
Site:     Lacq Gas Processing Site (illustrative)
Sector:   petrochemical
Location: Nouvelle-Aquitaine, France
Source:   Lacq basin, TotalEnergies Nouvelle-Aquitaine. TotalEnergies SR 2023.


In [3]:
# ── FUNCTIONS ─────────────────────────────────────────────────────────────────
# Redefined here from previous notebooks. Standard practice in Jupyter.

def calculate_scope1(site, ef_scope1):
    """Calculate Scope 1 emissions from direct fuel combustion."""
    results = {}
    fuel_keys = {
        'natural_gas': 'natural_gas_MWh',
        'fuel_oil':    'fuel_oil_MWh',
        'coal':        'coal_MWh',
    }
    for fuel, site_key in fuel_keys.items():
        if site_key in site:
            results[fuel] = round(site[site_key] * ef_scope1[fuel], 1)
    if 'process_tCO2' in site:
        results['process_emissions'] = site['process_tCO2']
    results['TOTAL_scope1'] = round(sum(results.values()), 1)
    return results

def calculate_scope2(site, ef_electricity):
    """Calculate Scope 2 emissions from purchased electricity."""
    grid    = site['grid']
    ef      = ef_electricity[grid]
    ef_ren  = ef_electricity['renewable_ppa']
    mwh     = site['electricity_MWh']
    return {
        'location_based': round(mwh * ef, 1),
        'market_based':   round(mwh * ef_ren, 1),
        'grid_ef':        ef,
    }

def scenario_ppa(site, ef_electricity, ppa_fraction):
    """Model switching a fraction of grid electricity to renewable PPA."""
    assert 0.0 <= ppa_fraction <= 1.0
    grid            = site['grid']
    mwh             = site['electricity_MWh']
    ef_grid         = ef_electricity[grid]
    ef_ppa          = ef_electricity['renewable_ppa']
    ef_new          = (ppa_fraction * ef_ppa) + ((1 - ppa_fraction) * ef_grid)
    scope2_baseline = mwh * ef_grid
    scope2_new      = mwh * ef_new
    return {
        'scope2_new_tCO2': round(scope2_new, 1),
        'reduction_tCO2':  round(scope2_baseline - scope2_new, 1),
    }

def scenario_biomethane(site, ef_scope1, ef_biomethane,
                        bio_fraction, feedstock='agricultural_waste'):
    """Replace a fraction of natural gas with biomethane."""
    assert 0.0 <= bio_fraction <= 1.0
    ng_mwh        = site.get('natural_gas_MWh', 0)
    scope1_ng_new = ng_mwh * (1 - bio_fraction) * ef_scope1['natural_gas']
    scope1_bio    = ng_mwh * bio_fraction * ef_biomethane[feedstock]
    scope1_oil    = site.get('fuel_oil_MWh', 0) * ef_scope1['fuel_oil']
    scope1_new    = scope1_ng_new + scope1_bio + scope1_oil
    scope1_base   = (ng_mwh * ef_scope1['natural_gas'] +
                     site.get('fuel_oil_MWh', 0) * ef_scope1['fuel_oil'])
    return {
        'scope1_new_tCO2': round(scope1_new, 1),
        'reduction_tCO2':  round(scope1_base - scope1_new, 1),
        'reduction_pct':   round((scope1_base - scope1_new) / scope1_base * 100, 1),
    }

def scenario_ccs(scope1_residual_tCO2, capture_rate=0.90):
    """Apply CCS to residual Scope 1 emissions."""
    assert 0.0 <= capture_rate <= 1.0
    return {
        'captured_tCO2':  round(scope1_residual_tCO2 * capture_rate, 1),
        'residual_tCO2':  round(scope1_residual_tCO2 * (1 - capture_rate), 1),
        'cost_EUR':       round(scope1_residual_tCO2 * capture_rate * 100, 0),
    }

def build_trajectory(site, ef_scope1, ef_elec, ef_bio, deployment_plan):
    """Build annual emissions trajectory from a deployment plan."""
    scope1_base = (site.get('natural_gas_MWh', 0) * ef_scope1['natural_gas'] +
                   site.get('fuel_oil_MWh', 0) * ef_scope1['fuel_oil'])
    scope2_base = site['electricity_MWh'] * ef_elec[site['grid']]
    baseline    = scope1_base + scope2_base
    rows = []
    for year in sorted(deployment_plan.keys()):
        plan     = deployment_plan[year]
        scope2_y = scenario_ppa(site, ef_elec, plan.get('ppa', 0))['scope2_new_tCO2']
        scope1_y = scenario_biomethane(site, ef_scope1, ef_bio,
                                       plan.get('bio', 0))['scope1_new_tCO2']
        if plan.get('ccs', False):
            scope1_y = scenario_ccs(scope1_y)['residual_tCO2']
        total_y = scope1_y + scope2_y
        rows.append({
            'year':          year,
            'scope1':        round(scope1_y, 0),
            'scope2':        round(scope2_y, 0),
            'total':         round(total_y, 0),
            'reduction_pct': round((baseline - total_y) / baseline * 100, 1),
        })
    return pd.DataFrame(rows).set_index('year'), baseline

print("All functions loaded.")

All functions loaded.


In [5]:
# ── BASELINE ──────────────────────────────────────────────────────────────────
scope1 = calculate_scope1(LACQ, SCOPE1)
scope2 = calculate_scope2(LACQ, SCOPE2_ELECTRICITY)

baseline_total = scope1['TOTAL_scope1'] + scope2['location_based']

print("LACQ BASELINE EMISSIONS")
print("=" * 45)
print(f"\nScope 1 — Direct combustion:")
for k, v in scope1.items():
    if k != 'TOTAL_scope1':
        pct = v / baseline_total * 100
        print(f"  {k:<20} {v:>10,.0f} tCO2eq  ({pct:.1f}%)")
print(f"  {'TOTAL Scope 1':<20} {scope1['TOTAL_scope1']:>10,.0f} tCO2eq")

print(f"\nScope 2 — Purchased electricity:")
print(f"  {'Location-based':<20} {scope2['location_based']:>10,.0f} tCO2eq  "
      f"({scope2['location_based']/baseline_total*100:.1f}%)")
print(f"  {'Market-based (PPA)':<20} {scope2['market_based']:>10,.0f} tCO2eq")

print(f"\n{'TOTAL BASELINE':<20} {baseline_total:>10,.0f} tCO2eq/year")
print(f"\nGrid emission factor: {scope2['grid_ef']} kgCO2eq/kWh (French nuclear grid)")

LACQ BASELINE EMISSIONS

Scope 1 — Direct combustion:
  natural_gas             181,600 tCO2eq  (85.8%)
  fuel_oil                 16,200 tCO2eq  (7.7%)
  TOTAL Scope 1           197,800 tCO2eq

Scope 2 — Purchased electricity:
  Location-based           13,750 tCO2eq  (6.5%)
  Market-based (PPA)        2,500 tCO2eq

TOTAL BASELINE          211,550 tCO2eq/year

Grid emission factor: 0.055 kgCO2eq/kWh (French nuclear grid)


In [7]:
# ── GRID COMPARISON: WHAT IF LACQ WERE IN ANOTHER COUNTRY? ───────────────────
# Same site, same electricity consumption, different grid.
# Shows why decarbonization strategy is geography-dependent.

mwh = LACQ['electricity_MWh']

grids = {
    'France (FR_2023)':      'FR_2023',
    'United Kingdom':        'UK_2023',
    'EU average':            'EU_avg_2023',
    'Germany (DE_2023)':     'DE_2023',
    'Renewable PPA':         'renewable_ppa',
}

print("SCOPE 2 COMPARISON — LACQ ON DIFFERENT ELECTRICITY GRIDS")
print("=" * 60)
print(f"\n{'Grid':<25} {'EF (kgCO2/kWh)':>15} {'Scope 2 (tCO2eq)':>18}")
print('-' * 60)

for label, key in grids.items():
    ef       = SCOPE2_ELECTRICITY[key]
    scope2   = mwh * ef
    marker   = ' ← actual' if key == 'FR_2023' else ''
    print(f"{label:<25} {ef:>15.3f} {scope2:>18,.0f}{marker}")

print()
print("Key insight: Lacq's Scope 2 on the German grid would be")
scope2_fr = mwh * SCOPE2_ELECTRICITY['FR_2023']
scope2_de = mwh * SCOPE2_ELECTRICITY['DE_2023']
print(f"{scope2_de/scope2_fr:.1f}x higher than on the French grid.")
print("This means PPA is a weak lever in France but a strong one in Germany.")

SCOPE 2 COMPARISON — LACQ ON DIFFERENT ELECTRICITY GRIDS

Grid                       EF (kgCO2/kWh)   Scope 2 (tCO2eq)
------------------------------------------------------------
France (FR_2023)                    0.055             13,750 ← actual
United Kingdom                      0.225             56,250
EU average                          0.255             63,750
Germany (DE_2023)                   0.380             95,000
Renewable PPA                       0.010              2,500

Key insight: Lacq's Scope 2 on the German grid would be
6.9x higher than on the French grid.
This means PPA is a weak lever in France but a strong one in Germany.


In [9]:
# ── SCENARIO COMPARISON ───────────────────────────────────────────────────────
# Run each lever independently at a meaningful adoption level.
# This shows the relative impact of each lever on total emissions.

ppa_100  = scenario_ppa(LACQ, SCOPE2_ELECTRICITY, ppa_fraction=1.00)
bio_40   = scenario_biomethane(LACQ, SCOPE1, BIOMETHANE,
                               bio_fraction=0.40,
                               feedstock='agricultural_waste')
ccs_90   = scenario_ccs(scope1['TOTAL_scope1'], capture_rate=0.90)

print("SCENARIO COMPARISON — INDIVIDUAL LEVERS")
print("=" * 55)
print(f"\nBaseline total: {baseline_total:,.0f} tCO2eq/year")
print()
print(f"{'Lever':<35} {'Reduction':>12} {'% of total':>11}")
print('-' * 60)
print(f"{'PPA 100% renewable electricity':<35} "
      f"{ppa_100['reduction_tCO2']:>12,.0f} "
      f"{ppa_100['reduction_tCO2']/baseline_total*100:>10.1f}%")
print(f"{'Biomethane 40% (agricultural waste)':<35} "
      f"{bio_40['reduction_tCO2']:>12,.0f} "
      f"{bio_40['reduction_tCO2']/baseline_total*100:>10.1f}%")
print(f"{'CCS 90% on Scope 1':<35} "
      f"{ccs_90['captured_tCO2']:>12,.0f} "
      f"{ccs_90['captured_tCO2']/baseline_total*100:>10.1f}%")
print()
print("Note: levers are shown independently.")
print("Combined effects are modelled in the trajectory below.")

SCENARIO COMPARISON — INDIVIDUAL LEVERS

Baseline total: 211,550 tCO2eq/year

Lever                                  Reduction  % of total
------------------------------------------------------------
PPA 100% renewable electricity            11,250        5.3%
Biomethane 40% (agricultural waste)       65,280       30.9%
CCS 90% on Scope 1                       178,020       84.2%

Note: levers are shown independently.
Combined effects are modelled in the trajectory below.


In [11]:
# ── TRAJECTORY AND SECTOR COMPARISON ─────────────────────────────────────────
LACQ_PLAN = {
    2024: {'ppa': 0.00, 'bio': 0.00, 'ccs': False},
    2026: {'ppa': 0.25, 'bio': 0.10, 'ccs': False},
    2028: {'ppa': 0.50, 'bio': 0.20, 'ccs': False},
    2030: {'ppa': 1.00, 'bio': 0.40, 'ccs': False},
    2035: {'ppa': 1.00, 'bio': 0.60, 'ccs': True },
    2040: {'ppa': 1.00, 'bio': 0.70, 'ccs': True },
    2050: {'ppa': 1.00, 'bio': 0.80, 'ccs': True },
}

# Build trajectory for Lacq
traj_lacq, baseline_lacq = build_trajectory(LACQ, SCOPE1, SCOPE2_ELECTRICITY,
                                             BIOMETHANE, LACQ_PLAN)

# Build trajectory for French gas sector average using same plan
traj_avg, baseline_avg = build_trajectory(FRENCH_GAS_SECTOR_AVG, SCOPE1,
                                           SCOPE2_ELECTRICITY, BIOMETHANE,
                                           LACQ_PLAN)

print("LACQ TRAJECTORY vs FRENCH GAS SECTOR AVERAGE")
print("=" * 65)
print()
print(f"{'Year':<6} {'Lacq total':>12} {'Lacq -%%':>10} {'Sector avg':>12} {'Avg -%%':>10}")
print('-' * 55)

for year in traj_lacq.index:
    lacq_total = traj_lacq.loc[year, 'total']
    lacq_pct   = traj_lacq.loc[year, 'reduction_pct']
    avg_total  = traj_avg.loc[year, 'total']
    avg_pct    = traj_avg.loc[year, 'reduction_pct']
    print(f"{year:<6} {lacq_total:>12,.0f} {lacq_pct:>9.1f}% "
          f"{avg_total:>12,.0f} {avg_pct:>9.1f}%")

print()
target_2030 = baseline_lacq * 0.45
print(f"Lacq baseline:          {baseline_lacq:>10,.0f} tCO2eq/year")
print(f"Sector avg baseline:    {baseline_avg:>10,.0f} tCO2eq/year")
print(f"EU 2030 target (Lacq):  {target_2030:>10,.0f} tCO2eq/year")
print(f"Lacq 2030 modelled:     {traj_lacq.loc[2030,'total']:>10,.0f} tCO2eq/year")
gap = traj_lacq.loc[2030, 'total'] - target_2030
print(f"Gap to 2030 target:     {gap:>10,.0f} tCO2eq  (plan needs strengthening)")

LACQ TRAJECTORY vs FRENCH GAS SECTOR AVERAGE

Year     Lacq total   Lacq -%%   Sector avg    Avg -%%
-------------------------------------------------------
2024        211,550       0.0%      157,440       0.0%
2026        192,418       9.0%      143,175       9.1%
2028        173,285      18.1%      128,910      18.1%
2030        135,020      36.2%      100,380      36.2%
2035         12,488      94.1%        9,210      94.2%
2040         10,856      94.9%        7,986      94.9%
2050          9,224      95.6%        6,762      95.7%

Lacq baseline:             211,550 tCO2eq/year
Sector avg baseline:       157,440 tCO2eq/year
EU 2030 target (Lacq):      95,198 tCO2eq/year
Lacq 2030 modelled:        135,020 tCO2eq/year
Gap to 2030 target:         39,822 tCO2eq  (plan needs strengthening)
